In [1]:
import pulp as pl
import networkx as nx

#### Model definition

In [2]:
b_chr_model = pl.LpProblem(name="b-chromatic-model")

#### Variables definition

In [3]:
n = 10 #number of vertices
V = range(n)
c = 5 #number of colours
C = range(c)

In [4]:
x_vars = {(v,c):pl.LpVariable(
                             cat=pl.LpBinary,
                             name="x_{0}_{1}".format(v,c))
            for v in V for c in C}

z_vars = {(v,c):pl.LpVariable(
                             cat=pl.LpBinary,
                             name="z_{0}_{1}".format(v,c))
            for v in V for c in C}

d_vars = {c:pl.LpVariable(
                cat=pl.LpBinary,
                name="d_{0}".format(c))
             for c in C}

#### Constraints

$\sum_{c\in C}x_{vc}=1\hspace{1cm}\forall v\in V$

In [9]:
one_colour_constraint = {v:b_chr_model.addConstraint(
                            pl.LpConstraint(
                                e=pl.lpSum(x_vars[v,c] for c in C),
                                sense=pl.LpConstraintEQ,
                                rhs=1,
                                name="one_colour_{}".format(v))
                        ) for v in V}

$x_{vc}+x_{wc}\leq 1\hspace{1cm}\forall vw\in E, \forall c\in C$

In [16]:
diff_colour_per_edge_constraint = {(v,w,c):b_chr_model.addConstraint(
                                            pl.LpConstraint(
                                                e=x_vars[v,c]+x_vars[w,c],
                                                sense=pl.LpConstraintLE,
                                                rhs=1,
                                                name="diff_colour_{0}_{1}_{2}".format(v,w,c)
                                            )) 
                                   for v in V for w in V for c in C}

$z_{vc}\leq \sum_{w\in N(v)}x_{wd}\hspace{1cm} \forall v\in V,\forall c,d\in C, c\neq d$

In [21]:
b_chromatic_vertex_constraint = {(v,c,d):b_chr_model.addConstraint(
                                            pl.LpConstraint(
                                                e=z_vars[v,c],
                                                sense=pl.LpConstraintLE,
                                                rhs=pl.lpSum(x_vars[w,d] for w in V),
                                                name="b_chr_vertex_{0}_{1}_{2}".format(v,c,d)
                                            )
                                        )
                                 for v in V for c in C for d in C if c!=d}

$d_c\leq \sum_{v\in V}z_{vc}\hspace{1cm} c\in C$

In [23]:
b_chromatic_vertices_constraint = {c:b_chr_model.addConstraint(
                                        pl.LpConstraint(
                                            e=d_vars[c],
                                            sense=pl.LpConstraintLE,
                                            rhs=pl.lpSum(z_vars[v,c] for v in V),
                                            name="b_chro_colour_{0}".format(c)
                                        )
                                    )
                                  for c in C}